# Isolation Forest — Anomali Tespiti\nAnormal noktaları erken izole etme prensibi: normal noktalar derin ağaç, anomaliler sığ ağaç gerektirir.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, seaborn as sns\nfrom sklearn.ensemble import IsolationForest\nfrom sklearn.datasets import make_blobs\nimport os; os.makedirs('cikti', exist_ok=True)

## Sentetik Veri ve Eğitim

In [ ]:
X, _ = make_blobs(n_samples=300, centers=1, cluster_std=1.0, random_state=42)\n# 30 anomali noktası ekle\nanomalies = np.random.uniform(low=-10, high=10, size=(30, 2))\nX = np.vstack([X, anomalies])\n\nmodel = IsolationForest(contamination=0.1, random_state=42)\npreds = model.fit_predict(X)\n\nnormal = X[preds == 1]; outlier = X[preds == -1]\nprint(f'Normal: {len(normal)}, Anomali: {len(outlier)}')\n\nplt.figure(figsize=(8, 6))\nplt.scatter(normal[:, 0], normal[:, 1], c='blue', alpha=0.5, label='Normal')\nplt.scatter(outlier[:, 0], outlier[:, 1], c='red', alpha=0.7, label='Anomali', edgecolors='black')\nplt.title(f'Isolation Forest (contamination=0.1)')\nplt.legend(); plt.savefig('cikti/isolation_forest.png', dpi=100, bbox_inches='tight')\nplt.show()

## Anomali Skorları

In [ ]:
scores = model.decision_function(X)\nanomali_skoru = -scores\n\nplt.figure(figsize=(10, 4))\nplt.subplot(1, 2, 1)\nplt.scatter(X[:, 0], X[:, 1], c=anomali_skoru, cmap='hot', alpha=0.7)\nplt.colorbar(label='Anomali Skoru'); plt.title('Skor Dağılımı')\n\nplt.subplot(1, 2, 2)\nplt.hist(anomali_skoru, bins=30, color='orange', edgecolor='black', alpha=0.7)\nplt.axvline(np.percentile(anomali_skoru, 90), color='red', linestyle='--', label='90. yüzdelik')\nplt.xlabel('Anomali Skoru'); plt.legend(); plt.title('Skor Histogramı')\nplt.tight_layout(); plt.savefig('cikti/isolation_forest_skor.png', dpi=100)\nplt.show()\nprint(f'En yüksek 5 anomali skoru: {np.sort(anomali_skoru)[-5:]}')

## Kredi Kartı Veri Seti Uygulaması

In [ ]:
# Kaggle Credit Card Fraud veri seti ile test (yoksa sentetik veri)\ntry:\n    import pandas as pd\n    df = pd.read_csv('creditcard.csv')\n    print(f'Veri yüklendi: {len(df)} işlem')\n    X_real = df.drop(['Time', 'Class'], axis=1).values[:5000]\n    y_real = df['Class'].values[:5000]\nexcept:\n    print('creditcard.csv bulunamadı, sentetik veri kullanılıyor.')\n    X_real = np.random.randn(1000, 5)\n    y_real = np.zeros(1000)\n    y_real[:20] = 1\n\nmodel2 = IsolationForest(contamination=0.02, random_state=42)\npreds_real = model2.fit_predict(X_real)\nanomali_bulunan = (preds_real == -1).sum()\nprint(f'Tespit edilen anomali: {anomali_bulunan} / {len(X_real)} (%{100*anomali_bulunan/len(X_real):.1f})')